In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Load Packages 
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import sys
sys.path.append('/project/IZZY/MolRepres/Methods/')                  # Path where the training, evaluating function is located 
from torch_geometric.datasets import QM9                             # Dataset utilities for 3D molecular graphs (QM9 with 3D coordinates)
from torch_geometric.nn import SAGEConv, global_mean_pool             # Graph Neural Network layers and pooling operations from PyTorch Geometric
from torch_geometric.loader import DataLoader                        # DataLoader to batch and shuffle molecular graph data
from torch.optim import Adam                                         # Optimizer (Adam) and learning rate scheduler (StepLR)
from torch.optim.lr_scheduler import StepLR                          # Progress bar for training/evaluation loops
from tqdm import tqdm                                                # Unit conversion constants (Hartree, eV, Bohr, Angstrom) from ASE
from ase.units import Hartree, eV, Bohr, Ang                         # TensorBoard writer for tracking metrics and visualizing training progress
from torch.utils.tensorboard import SummaryWriter                    # TensorBoard writer for tracking metrics and visualizing training progress
from sklearn.preprocessing import StandardScaler                     # Tool for normalizing data (zero mean, unit variance)
from train_eval import train_epoch, evaluate                         # Importing the training and evaluating function

In [3]:
# GPU Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # This line checks if a CUDA-enabled GPU is available. If yes, computations will be performed on the GPU for faster training. Otherwise, it falls back to using the CPU.
# Print which device is being used 
print("Using device:", device)

Using device: cuda


In [4]:
# Load dataset and split
# Load the QM9 dataset 
dataset = QM9(root='data/QM9')

# Count how many molecular graphs are in the dataset
num_mols = len(dataset)
print(f"Total QM9 molecules: {num_mols}")

# Define the desired split sizes for training, validation, and testing
train_size, valid_size = 110000, 10000
# Ensure the sum of train and validation sizes does not exceed dataset size
assert train_size + valid_size < num_mols, "Split sizes too large for dataset"

# Randomly shuffle all molecule indices
perm = torch.randperm(num_mols, generator=torch.Generator().manual_seed(42))

# Use the shuffled indices to create non-overlapping subsets
train_idx = perm[:train_size]
valid_idx = perm[train_size:train_size+valid_size]
test_idx  = perm[train_size+valid_size:]

# Build subsets based on the index splits
train_dataset = dataset[train_idx]
valid_dataset = dataset[valid_idx]
test_dataset  = dataset[test_idx]

Total QM9 molecules: 130831


In [5]:
# Define the QM9 Targerts
PROPERTY_NAMES = [
    'μ (D)', 'α (Ang³)', 'ε_HOMO (eV)', 'ε_LUMO (eV)', 'Δε (eV)', '⟨R²⟩ (Ang²)',
    'ZPVE (eV)', 'U₀ (eV)', 'U (eV)', 'H (eV)', 'G (eV)', 'c_v', 'U₀_atom',
    'U_atom', 'H_atom', 'G_atom', 'A (GHz)', 'B (GHz)', 'C (GHz)'
] 

# QM9 Unit Conversion Factors
def get_qm9_conversions_tensor(device):
    return torch.tensor([
        1.0,                # 0 - μ (D)     
        Bohr**3 / Ang**3,   # 1 - α (Bohr³ → Å³)
        Hartree / eV,       # 2 - ε_HOMO
        Hartree / eV,       # 3 - ε_LUMO
        Hartree / eV,       # 4 - Δε
        Bohr**2 / Ang**2,   # 5 - ⟨R²⟩
        Hartree / eV,       # 6 - ZPVE
        Hartree / eV,       # 7 - U₀
        Hartree / eV,       # 8 - U
        Hartree / eV,       # 9 - H
        Hartree / eV,       # 10 - G
        1.0,                # 11 - c_v
        1.0,                # 12 - U₀_atom
        Hartree / eV,       # 13 - U_atom
        Hartree / eV,       # 14 - H_atom
        Hartree / eV,       # 15 - G_atom
        1.0,                # 16 - A (GHz)
        1.0,                # 17 - B (GHz)
        1.0                 # 18 - C (GHz)
    ], dtype=torch.float, device=device)

In [6]:
# Normalize all QM9 targets
y_raw_all = dataset.data.y.clone().cpu()                
conversions_cpu = get_qm9_conversions_tensor('cpu')    
y_conv_all = y_raw_all * conversions_cpu.unsqueeze(0)   

norm_stats = {'mean': [], 'std': []}
y_norm_all = torch.zeros_like(y_conv_all)

for i in range(y_conv_all.shape[1]):
    train_y_cpu = y_conv_all[train_idx.cpu(), i]
    mean_i = float(train_y_cpu.mean().item())
    std_i = float(train_y_cpu.std().item()) if train_y_cpu.std().item() != 0 else 1.0
    y_norm_all[:, i] = (y_conv_all[:, i] - mean_i) / std_i
    norm_stats['mean'].append(mean_i)
    norm_stats['std'].append(std_i)

dataset.data.y = y_norm_all.to(torch.float)
print("Normalization complete for all targets.")

Normalization complete for all targets.


/home/ismail/dig_envi/lib/python3.9/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [7]:
# Build GraphSAGE-Model 
in_channels = dataset.num_node_features
print("Node feature dim:", in_channels)

# Define SAGE-Model
class SAGEModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        # Three graph convolutional layers
        self.conv1 = SAGEConv(in_dim, hidden_dim)                  # Each layer updates node embeddings using neighbors' features
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, hidden_dim)

        # Fully connected layers to map pooled graph embedding => target value
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 19)                               # Output: single scalar target
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, batch):

        x, edge_index, batch_vec = batch.x, batch.edge_index, batch.batch

        # Apply three SAGE layers with ReLU activation
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        
        # Global mean Pooling
        x = global_mean_pool(x, batch_vec)       # The global mean pool ing takes the mean of all atom embeddings in each molecule´s graph "one feat vector per molecule"
        x = F.relu(self.fc1(x)) 
        x = self.dropout(x)
        return self.fc2(x)

Node feature dim: 11


In [8]:
# Trainig Loop
def main():
    # Hyperparameters
    epochs = 1000                            # number of training epochs
    batch_size = 128                        # batch size for training
    vt_batch_size = 256                     # batch size for validation
    lr = 1e-5                               # learning rate
    lr_decay_step = 50                      # steps after which LR is decayed
    lr_decay_factor = 0.5                   # factor to decay learning rate
    weight_decay = 1e-4                     # L2 regularization
    save_dir = 'checkpoints_SAGE'            # directory to save model checkpoints 
    log_dir = 'logs_SAGE'                    # TensorBoard logs directory

    #Create directories if they don't exist
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    # TensorBoard for visualization of metrics
    writer = SummaryWriter(log_dir=log_dir)

    # Shuffled batches for training; sequential batches for validation/testing
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(valid_dataset, batch_size=vt_batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=vt_batch_size, shuffle=False)

    # SAGE model
    model = SAGEModel(in_channels).to(device) 

    # Optimizer and Scheduler
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer, step_size=lr_decay_step, gamma=lr_decay_factor)

    # Track best validation/test performance
    best_mean_val=float('inf')
    best_val = [float('inf')] * len(PROPERTY_NAMES)
    best_test = [float('inf')] * len(PROPERTY_NAMES)

    # Print total number of trainabel parameters and target property
    print("#params:", sum(p.numel() for p in model.parameters()))
    print("Training for all 19 targets") 

    # Training Loop
    for epoch in range(1, epochs + 1):
        print(f"\n=== Epoch {epoch} ===")

        #Train for one epoch
        train_loss = train_epoch(model, train_loader, optimizer, device)

        # Evaluate on validation and test sets (MAE in original units)
        val_mae = evaluate(model, val_loader, device, norm_stats['mean'], norm_stats['std'])
        test_mae = evaluate(model, test_loader, device, norm_stats['mean'], norm_stats['std'])

        # Print epoch metrics
        print(f"Train loss (MSE): {train_loss:.6f}")
        for i, prop in enumerate(PROPERTY_NAMES):
            print(f"  {prop:15s} | Val MAE: {val_mae[i]:.6f} | Test MAE: {test_mae[i]:.6f}")
            writer.add_scalar(f'val_mae/{prop}', val_mae[i], epoch)
            writer.add_scalar(f'test_mae/{prop}', test_mae[i], epoch)

        # Save checkpoint if validation improves 
        mean_val_mae = sum(val_mae) / len(val_mae)
        
        # If mean validation MAE improved → save one best model
        if mean_val_mae < best_mean_val:
            best_mean_val = mean_val_mae
            best_val = val_mae
            best_test = test_mae
        
            save_path = os.path.join(save_dir, 'best_model.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val': best_val,
                'best_test': best_test,
                'best_mean_val': best_mean_val
            }, save_path)
        
            print(f" Saved best overall model (mean validation MAE improved to {best_mean_val:.4f})")

        # Update learning rate according to scheduler
        scheduler.step()
    # Close TensorBoard writer
    writer.close()
    print("\nFinished training.")
    print("Best validation and test MAEs per property:")
    for prop, v, t in zip(PROPERTY_NAMES, best_val, best_test):
        print(f"  {prop:15s} | Best validation MAE: {v:.6f} | Test MAE at best val: {t:.6f}")

    #print("Best validation MAE:", best_val)
    #print("Test MAE at best val:", best_test)

# Run the training loop
if __name__ == "__main__":
    main()

#params: 78227
Training for all 19 targets

=== Epoch 1 ===


Train loss (MSE): 0.988681
  μ (D)           | Val MAE: 1.136485 | Test MAE: 1.155802
  α (Ang³)        | Val MAE: 0.894859 | Test MAE: 0.890551
  ε_HOMO (eV)     | Val MAE: 11.955853 | Test MAE: 12.006211
  ε_LUMO (eV)     | Val MAE: 27.229656 | Test MAE: 27.636335
  Δε (eV)         | Val MAE: 28.549852 | Test MAE: 28.665016
  ⟨R²⟩ (Ang²)     | Val MAE: 55.875336 | Test MAE: 55.847511
  ZPVE (eV)       | Val MAE: 18.300127 | Test MAE: 18.250597
  U₀ (eV)         | Val MAE: 22413.951172 | Test MAE: 22055.365234
  U (eV)          | Val MAE: 22350.869141 | Test MAE: 21992.605469
  H (eV)          | Val MAE: 23204.863281 | Test MAE: 22810.419922
  G (eV)          | Val MAE: 22427.597656 | Test MAE: 22062.132812
  c_v             | Val MAE: 3.064780 | Test MAE: 3.034835
  U₀_atom         | Val MAE: 7.833310 | Test MAE: 7.803960
  U_atom          | Val MAE: 208.318939 | Test MAE: 207.312622
  H_atom          | Val MAE: 223.368118 | Test MAE: 222.597549
  G_atom          | Val MAE: 184.08647

Train loss (MSE): 0.863936
  μ (D)           | Val MAE: 1.091514 | Test MAE: 1.110945
  α (Ang³)        | Val MAE: 0.722070 | Test MAE: 0.715101
  ε_HOMO (eV)     | Val MAE: 12.016814 | Test MAE: 12.053373
  ε_LUMO (eV)     | Val MAE: 23.440945 | Test MAE: 23.641071
  Δε (eV)         | Val MAE: 25.280090 | Test MAE: 25.241678
  ⟨R²⟩ (Ang²)     | Val MAE: 53.645473 | Test MAE: 53.578529
  ZPVE (eV)       | Val MAE: 10.291498 | Test MAE: 10.091104
  U₀ (eV)         | Val MAE: 20872.152344 | Test MAE: 20514.998047
  U (eV)          | Val MAE: 20161.888672 | Test MAE: 19838.488281
  H (eV)          | Val MAE: 21742.498047 | Test MAE: 21379.306641
  G (eV)          | Val MAE: 20611.117188 | Test MAE: 20259.222656
  c_v             | Val MAE: 2.415227 | Test MAE: 2.377905
  U₀_atom         | Val MAE: 5.148540 | Test MAE: 5.072847
  U_atom          | Val MAE: 125.540741 | Test MAE: 122.911331
  H_atom          | Val MAE: 164.783920 | Test MAE: 162.773254
  G_atom          | Val MAE: 108.06040

Train loss (MSE): 0.730380
  μ (D)           | Val MAE: 1.058299 | Test MAE: 1.077109
  α (Ang³)        | Val MAE: 0.674765 | Test MAE: 0.666644
  ε_HOMO (eV)     | Val MAE: 11.873930 | Test MAE: 11.923792
  ε_LUMO (eV)     | Val MAE: 22.873077 | Test MAE: 23.054586
  Δε (eV)         | Val MAE: 24.362749 | Test MAE: 24.305782
  ⟨R²⟩ (Ang²)     | Val MAE: 53.426483 | Test MAE: 53.437119
  ZPVE (eV)       | Val MAE: 7.730369 | Test MAE: 7.506427
  U₀ (eV)         | Val MAE: 20084.912109 | Test MAE: 19740.212891
  U (eV)          | Val MAE: 19820.169922 | Test MAE: 19499.287109
  H (eV)          | Val MAE: 20147.039062 | Test MAE: 19816.031250
  G (eV)          | Val MAE: 19973.001953 | Test MAE: 19636.591797
  c_v             | Val MAE: 2.330635 | Test MAE: 2.293360
  U₀_atom         | Val MAE: 4.047242 | Test MAE: 3.945672
  U_atom          | Val MAE: 101.917366 | Test MAE: 98.611374
  H_atom          | Val MAE: 118.901688 | Test MAE: 115.880508
  G_atom          | Val MAE: 90.478851 | 

Train loss (MSE): 0.694657
  μ (D)           | Val MAE: 1.040447 | Test MAE: 1.059325
  α (Ang³)        | Val MAE: 0.657234 | Test MAE: 0.648911
  ε_HOMO (eV)     | Val MAE: 11.811583 | Test MAE: 11.874120
  ε_LUMO (eV)     | Val MAE: 22.561527 | Test MAE: 22.752441
  Δε (eV)         | Val MAE: 24.027401 | Test MAE: 24.010674
  ⟨R²⟩ (Ang²)     | Val MAE: 53.394569 | Test MAE: 53.489052
  ZPVE (eV)       | Val MAE: 7.028444 | Test MAE: 6.796638
  U₀ (eV)         | Val MAE: 19619.810547 | Test MAE: 19267.107422
  U (eV)          | Val MAE: 19542.117188 | Test MAE: 19210.798828
  H (eV)          | Val MAE: 19631.410156 | Test MAE: 19293.042969
  G (eV)          | Val MAE: 19591.468750 | Test MAE: 19248.177734
  c_v             | Val MAE: 2.350379 | Test MAE: 2.312484
  U₀_atom         | Val MAE: 3.628370 | Test MAE: 3.507539
  U_atom          | Val MAE: 92.818993 | Test MAE: 89.067894
  H_atom          | Val MAE: 101.751022 | Test MAE: 98.229736
  G_atom          | Val MAE: 82.222649 | Te

...

Train loss (MSE): 0.484159
  μ (D)           | Val MAE: 0.887224 | Test MAE: 0.895593
  α (Ang³)        | Val MAE: 0.496316 | Test MAE: 0.485769
  ε_HOMO (eV)     | Val MAE: 9.361321 | Test MAE: 9.272482
  ε_LUMO (eV)     | Val MAE: 15.006161 | Test MAE: 15.081339
  Δε (eV)         | Val MAE: 18.936745 | Test MAE: 18.784225
  ⟨R²⟩ (Ang²)     | Val MAE: 44.544338 | Test MAE: 43.993710
  ZPVE (eV)       | Val MAE: 5.242782 | Test MAE: 5.024879
  U₀ (eV)         | Val MAE: 11769.885742 | Test MAE: 11378.333008
  U (eV)          | Val MAE: 11725.665039 | Test MAE: 11335.561523
  H (eV)          | Val MAE: 11821.117188 | Test MAE: 11430.227539
  G (eV)          | Val MAE: 11834.718750 | Test MAE: 11441.843750
  c_v             | Val MAE: 1.710880 | Test MAE: 1.649361
  U₀_atom         | Val MAE: 3.153246 | Test MAE: 3.037567
  U_atom          | Val MAE: 86.498276 | Test MAE: 83.345078
  H_atom          | Val MAE: 86.696724 | Test MAE: 83.504097
  G_atom          | Val MAE: 79.554375 | Test 

Train loss (MSE): 0.484376
  μ (D)           | Val MAE: 0.887224 | Test MAE: 0.895593
  α (Ang³)        | Val MAE: 0.496316 | Test MAE: 0.485769
  ε_HOMO (eV)     | Val MAE: 9.361321 | Test MAE: 9.272482
  ε_LUMO (eV)     | Val MAE: 15.006160 | Test MAE: 15.081338
  Δε (eV)         | Val MAE: 18.936743 | Test MAE: 18.784224
  ⟨R²⟩ (Ang²)     | Val MAE: 44.544342 | Test MAE: 43.993710
  ZPVE (eV)       | Val MAE: 5.242782 | Test MAE: 5.024879
  U₀ (eV)         | Val MAE: 11769.885742 | Test MAE: 11378.333008
  U (eV)          | Val MAE: 11725.665039 | Test MAE: 11335.561523
  H (eV)          | Val MAE: 11821.117188 | Test MAE: 11430.226562
  G (eV)          | Val MAE: 11834.718750 | Test MAE: 11441.843750
  c_v             | Val MAE: 1.710880 | Test MAE: 1.649361
  U₀_atom         | Val MAE: 3.153246 | Test MAE: 3.037567
  U_atom          | Val MAE: 86.498276 | Test MAE: 83.345078
  H_atom          | Val MAE: 86.696724 | Test MAE: 83.504082
  G_atom          | Val MAE: 79.554359 | Test 

Train loss (MSE): 0.484396
  μ (D)           | Val MAE: 0.887224 | Test MAE: 0.895593
  α (Ang³)        | Val MAE: 0.496316 | Test MAE: 0.485769
  ε_HOMO (eV)     | Val MAE: 9.361321 | Test MAE: 9.272482
  ε_LUMO (eV)     | Val MAE: 15.006161 | Test MAE: 15.081339
  Δε (eV)         | Val MAE: 18.936745 | Test MAE: 18.784225
  ⟨R²⟩ (Ang²)     | Val MAE: 44.544338 | Test MAE: 43.993710
  ZPVE (eV)       | Val MAE: 5.242782 | Test MAE: 5.024879
  U₀ (eV)         | Val MAE: 11769.884766 | Test MAE: 11378.333008
  U (eV)          | Val MAE: 11725.665039 | Test MAE: 11335.561523
  H (eV)          | Val MAE: 11821.117188 | Test MAE: 11430.226562
  G (eV)          | Val MAE: 11834.718750 | Test MAE: 11441.842773
  c_v             | Val MAE: 1.710880 | Test MAE: 1.649361
  U₀_atom         | Val MAE: 3.153246 | Test MAE: 3.037567
  U_atom          | Val MAE: 86.498276 | Test MAE: 83.345078
  H_atom          | Val MAE: 86.696724 | Test MAE: 83.504082
  G_atom          | Val MAE: 79.554359 | Test 

Train loss (MSE): 0.484224
  μ (D)           | Val MAE: 0.887224 | Test MAE: 0.895593
  α (Ang³)        | Val MAE: 0.496316 | Test MAE: 0.485769
  ε_HOMO (eV)     | Val MAE: 9.361321 | Test MAE: 9.272482
  ε_LUMO (eV)     | Val MAE: 15.006160 | Test MAE: 15.081339
  Δε (eV)         | Val MAE: 18.936743 | Test MAE: 18.784225
  ⟨R²⟩ (Ang²)     | Val MAE: 44.544338 | Test MAE: 43.993710
  ZPVE (eV)       | Val MAE: 5.242783 | Test MAE: 5.024879
  U₀ (eV)         | Val MAE: 11769.885742 | Test MAE: 11378.333008
  U (eV)          | Val MAE: 11725.665039 | Test MAE: 11335.561523
  H (eV)          | Val MAE: 11821.117188 | Test MAE: 11430.226562
  G (eV)          | Val MAE: 11834.718750 | Test MAE: 11441.842773
  c_v             | Val MAE: 1.710880 | Test MAE: 1.649361
  U₀_atom         | Val MAE: 3.153246 | Test MAE: 3.037567
  U_atom          | Val MAE: 86.498276 | Test MAE: 83.345078
  H_atom          | Val MAE: 86.696724 | Test MAE: 83.504082
  G_atom          | Val MAE: 79.554359 | Test 

Train loss (MSE): 0.484412
  μ (D)           | Val MAE: 0.887224 | Test MAE: 0.895593
  α (Ang³)        | Val MAE: 0.496316 | Test MAE: 0.485769
  ε_HOMO (eV)     | Val MAE: 9.361321 | Test MAE: 9.272482
  ε_LUMO (eV)     | Val MAE: 15.006160 | Test MAE: 15.081338
  Δε (eV)         | Val MAE: 18.936745 | Test MAE: 18.784225
  ⟨R²⟩ (Ang²)     | Val MAE: 44.544338 | Test MAE: 43.993710
  ZPVE (eV)       | Val MAE: 5.242783 | Test MAE: 5.024879
  U₀ (eV)         | Val MAE: 11769.885742 | Test MAE: 11378.333008
  U (eV)          | Val MAE: 11725.665039 | Test MAE: 11335.561523
  H (eV)          | Val MAE: 11821.117188 | Test MAE: 11430.226562
  G (eV)          | Val MAE: 11834.717773 | Test MAE: 11441.843750
  c_v             | Val MAE: 1.710880 | Test MAE: 1.649361
  U₀_atom         | Val MAE: 3.153246 | Test MAE: 3.037567
  U_atom          | Val MAE: 86.498276 | Test MAE: 83.345078
  H_atom          | Val MAE: 86.696724 | Test MAE: 83.504082
  G_atom          | Val MAE: 79.554375 | Test 